In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# 'TotalCharges' is recognized as an object because of blank spaces. Convert to float.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df = df.drop('customerID', axis=1)

# Convert target variable 'Churn' to binary (1 for Yes, 0 for No)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 3. Split the data into features (X) and target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# 4. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

# Create the preprocessing pipeline for numerical features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Fills missing TotalCharges with the median
    ('scaler', StandardScaler())                   # Scales data to have mean=0 and variance=1
])

# Create the preprocessing pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Fills any missing text values
    ('onehot', OneHotEncoder(handle_unknown='ignore'))    # Converts text to numerical binary columns
])

# Combine them using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [8]:
# Create the full pipeline with a placeholder classifier (we'll start with Logistic Regression)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)) 
])

In [9]:
# Define the parameter grid
# Note the 'classifier__' prefix. This tells GridSearchCV to apply these to the 'classifier' step.
param_grid = [
    {
        'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__solver': ['liblinear', 'lbfgs']
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 5, 10]
    }
]

# Set up GridSearchCV
print("Starting Grid Search. This might take a minute...")
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit it to the training data
grid_search.fit(X_train, y_train)

print(f"Best Parameters found: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

# Evaluate the best model on the completely unseen test set
test_accuracy = grid_search.score(X_test, y_test)
print(f"Test Set Accuracy: {test_accuracy:.4f}")

Starting Grid Search. This might take a minute...
Best Parameters found: {'classifier': LogisticRegression(max_iter=1000, random_state=42), 'classifier__C': 10.0, 'classifier__solver': 'lbfgs'}
Best Cross-Validation Accuracy: 0.8051
Test Set Accuracy: 0.8055


In [10]:

import joblib
# Save the complete, fitted pipeline
model_filename = 'telco_churn_pipeline.pkl'
joblib.dump(grid_search.best_estimator_, model_filename)

print(f"Pipeline successfully saved to {model_filename}")

Pipeline successfully saved to telco_churn_pipeline.pkl
